# PolicyProof AI — Kaggle GPU recovery

**Use this if Colab has exhausted its GPU quota.** This notebook is compatible in principle with Kaggle T4 or P100 GPUs but real training has **not yet been validated on Kaggle**. All training is OFF by default. CuratorKIT data must be the original audited ZIP. Never upload Hugging Face tokens into code or a public dataset.

In Kaggle notebook Settings, enable Internet and select **Accelerator → GPU T4 x2** if available, otherwise **GPU P100**. This is a separate runtime and cannot recover your previous Colab in-memory model. Add your original `policyproof_curatorkit_train_v1.zip` through **Add Input** as a private Kaggle dataset, or upload it into `/kaggle/working`.

## 1. Verify GPU and install the known training stack (no model loading)

In [ ]:
import os, subprocess, sys, pathlib, torch
from importlib.metadata import version
assert torch.cuda.is_available(), "Enable the GPU accelerator under Kaggle Settings."
device=torch.cuda.get_device_name(0)
assert any(x in device for x in ["T4", "P100"]), f"Prepared for T4/P100, not {device}"
print("GPU:",device)
ROOT=pathlib.Path("/kaggle/working/PolicyProof-AI")
if not ROOT.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
subprocess.run([sys.executable,"-m","pip","install","--prefer-binary",
    "aligntune==0.1.28","trl==1.7.1","bitsandbytes==0.50.2"],check=True)
print({p:version(p) for p in ["aligntune","trl","bitsandbytes"]})


## 2. Locate the original CuratorKIT ZIP and authenticate to the gated model

The ZIP may be in a Kaggle private input dataset or `/kaggle/working`. Never substitute the 48-case development test or 60-case candidate final holdout for training. For gated Hugging Face access, create a **private** `HF_TOKEN` in Kaggle Secrets (Add-ons → Secrets), or use the Hugging Face login widget. Never print the token.

In [ ]:
from pathlib import Path
import glob, json
from huggingface_hub import login, model_info, notebook_login
from scripts.restore_colab_session import restore_curated_archive, EXPECTED_SHA
from scripts.check_align_config import check_train_data

CURATED=ROOT/"artifacts"/"verified_curator_train"
train_path=CURATED/"sft_alpaca.jsonl"
if train_path.exists():
    audited=check_train_data(train_path)
    assert audited["dataset_sha256"]==EXPECTED_SHA
    print("Existing curated training dataset verified.")
else:
    archives = list(Path("/kaggle/input").rglob("policyproof_curatorkit_train_v1.zip")) + list(Path("/kaggle/working").glob("policyproof_curatorkit_train_v1.zip"))
    assert len(archives) == 1, "Attach exactly one original CuratorKIT ZIP using Add Input (private), or upload into /kaggle/working."
    audited=restore_curated_archive(archives[0],CURATED)
print("Verified training rows:",audited["row_count"])

try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token=None
if token:
    login(token=token,add_to_git_credential=False)
try:
    MODEL_SHA=model_info("CohereLabs/tiny-aya-global").sha
except Exception:
    print("Model access is gated. Authenticate with a read-only Hugging Face token.")
    notebook_login()
    MODEL_SHA=model_info("CohereLabs/tiny-aya-global").sha
print("Approved model revision:",MODEL_SHA)


## 3. Load one 4-bit PEFT model and preflight all 78 tokenization samples

This step downloads and uses GPU memory, but does **not** perform training or evaluate the held-out policies.

In [ ]:
from scripts.restore_colab_session import restore_model_and_data
assert "trainer" not in globals(), "A model is already in memory. Do not load duplicates."
trainer,restore_report=restore_model_and_data(CURATED, expected_gpu=None)
print(json.dumps(restore_report,indent=2))
assert restore_report["tokenization"]["checked_rows"] == 78
print("RESTORE COMPLETE — zero optimizer steps.")


## 4. Optional direct PyTorch ten-step smoke training

Set `RUN_TRAINING=True` only after Step 3 succeeds and you are ready. This is the guarded **PyTorch fallback**, not an AlignTune `SFTTrainer` success, and it cannot establish safety improvement. The model and 4-bit adapters are loaded with AlignTune. A successful run writes `policyproof_t4_torch_fallback.zip` to `/kaggle/working` for download in the Files panel. Preserve failure reports rather than rerunning blindly.

In [ ]:
RUN_TRAINING=False
if RUN_TRAINING:
    from scripts.t4_torch_fallback import train_existing_t4
    archive=train_existing_t4(trainer,root=str(ROOT),model_sha=MODEL_SHA)
    assert archive.is_file()
    print("Verified training archive:",archive)
else:
    print("Training OFF. No model weights were optimized in this cell.")


## 5. Evidence and limitations

On success, download the ZIP from the Kaggle **Output/Files** panel and share it for integrity review. No before/after improvement is proven until the real adapter is evaluated on a new reviewed and frozen dataset with AuditKIT and manual semantic grading. If GPU access is unavailable here too, perform the CPU-only benchmark validation and writing tasks instead. See `docs/SUBMISSION_READINESS.md` and `docs/FINAL_BENCHMARK_PROTOCOL.md`.

## Optional: genuine 40-step QLoRA training run — separate from the 10-step smoke

Use this **instead of** the earlier 10-step AlignTune or PyTorch smoke training cells, with a freshly restored, never-trained 4-bit Tiny Aya model and the exact 78-row CuratorKIT training archive. It deliberately uses a distinct output folder and ZIP.

The first 66 deterministically selected **training-source** rows receive optimizer updates; the remaining 12 **training-source** rows are used only for a small held-out loss check after steps 10, 20, 30 and 40. None of the 48-case development benchmark, 60-case final candidate, or 72-case adversarial challenge is used for training or validation. This does NOT prove improved safety.

This path uses **direct PyTorch over an AlignTune-loaded PEFT model**, not a successful AlignTune trainer run. It avoids the TRL FP16/BF16 GradScaler incompatibility observed previously. It requires actual GPU access and a full pinned 40-hex base revision. Training is OFF by default. On success, save and securely download the adapter ZIP and training receipt; no benchmark result is claimed.

In [ ]:
RUN_EXTENDED_TRAINING = False  # Explicit opt-in only; starts CUDA optimizer steps
if RUN_EXTENDED_TRAINING:
    assert not globals().get("RUN_TRAINING", False), "Do not run smoke and extended training in one session"
    assert "trainer" in globals() and getattr(trainer, "model", None) is not None
    assert len(MODEL_SHA) == 40, "Pin the exact Hugging Face base revision"
    from scripts.t4_torch_fallback import training_plan, train_existing_t4
    print("TRAINING SCHEDULE:", {k: v for k, v in training_plan(steps=40).items()
          if k not in ("train_indices", "validation_indices")})
    archive = train_existing_t4(
        trainer, root=str(ROOT), model_sha=MODEL_SHA, optimizer_steps=40,
    )
    print("DOWNLOAD THE ARCHIVE FROM KAGGLE OUTPUT/FILES:", archive)
else:
    print("Extended training OFF: no GPU optimizer steps executed.")
